In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

import plotly.express as px
from gas_forecast.evaluation import error_metrics

sys.path.insert(0, str(Path("../src").resolve()))

from gas_forecast import evaluate_forecast, plot_weekly_change_forecast
from gas_forecast.models import (
    FiveYearWeeklyAverageModel,
    WeeklyChangeFourierRegressionModel,
    WeeklyChangeLinearRegressionModel,
    WeeklyChangeSARIMAModel,
)

STORAGE_PATH = Path("../datasets/processed/lower48_weekly_storage_latest.parquet")

storage = pd.read_parquet(STORAGE_PATH)
storage.head()

,date,total_storage_bcf,weekly_change_bcf,year,month,week_of_year,sequential_week_count
0,2014-07-04,2023,NaN,2014,7,27,1
1,2014-07-11,2128,105.0,2014,7,28,2
2,2014-07-18,2219,91.0,2014,7,29,3
3,2014-07-25,2307,88.0,2014,7,30,4
4,2014-08-01,2388,81.0,2014,8,31,5


Baseline (Naive) 5yr Weekly Average

In [55]:
# 5-Year Weekly Average
baseline_model = FiveYearWeeklyAverageModel(lookback_years=5)
baseline_result = evaluate_forecast(storage, baseline_model)
baseline_result.head()


,date,week_of_year,weekly_change_bcf,predicted_weekly_change,weekly_change_std,lower_band,upper_band,forecast_deviation,outside_band
600,2026-01-02,1,-120.0,-96.4,78.824489,-175.224489,-17.575511,-23.6,False
601,2026-01-09,2,-70.0,-177.4,65.297779,-242.697779,-112.102221,107.4,True
602,2026-01-16,3,-120.0,-196.4,93.382547,-289.782547,-103.017453,76.4,False
603,2026-01-23,4,-241.0,-225.8,67.843202,-293.643202,-157.956798,-15.2,False
604,2026-01-30,5,-359.0,-171.8,59.031348,-230.831348,-112.768652,-187.2,True


In [56]:
plot_weekly_change_forecast(baseline_result, model_name=baseline_model.name).show()

Linear Regression with cos/sin harmonic (to capture seasonality)

In [57]:
# Linear Regression (seasonal) - cos/sin = 1 harmonic
lr_model = WeeklyChangeLinearRegressionModel(lookback_years=5)
lr_result = evaluate_forecast(storage, lr_model)
lr_result.head()

,date,week_of_year,weekly_change_bcf,predicted_weekly_change,lower_band,upper_band,forecast_deviation,outside_band
600,2026-01-02,1,-120.0,-94.310804,-160.709985,-27.911623,-25.689196,False
601,2026-01-09,2,-70.0,-100.439280,-166.838461,-34.040099,30.439280,False
602,2026-01-16,3,-120.0,-105.612194,-172.011376,-39.213013,-14.387806,False
603,2026-01-23,4,-241.0,-109.754115,-176.153296,-43.354933,-131.245885,True
604,2026-01-30,5,-359.0,-112.804642,-179.203823,-46.405461,-246.195358,True


In [58]:
plot_weekly_change_forecast(lr_result, model_name=lr_model.name).show()

In [59]:
# Fourier Regression (K=3 harmonics)
fourier_model = WeeklyChangeFourierRegressionModel(lookback_years=5, n_harmonics=3)
fourier_result = evaluate_forecast(storage, fourier_model)
fourier_result.head()

,date,week_of_year,weekly_change_bcf,predicted_weekly_change,lower_band,upper_band,forecast_deviation,outside_band
600,2026-01-02,1,-120.0,-160.038961,-202.024707,-118.053214,40.038961,False
601,2026-01-09,2,-70.0,-169.548843,-211.534590,-127.563096,99.548843,True
602,2026-01-16,3,-120.0,-173.456699,-215.442446,-131.470952,53.456699,True
603,2026-01-23,4,-241.0,-171.701747,-213.687494,-129.716000,-69.298253,True
604,2026-01-30,5,-359.0,-164.488120,-206.473867,-122.502373,-194.511880,True


In [60]:
plot_weekly_change_forecast(fourier_result, model_name=fourier_model.name).show()

In [ ]:
# Fourier testing with different harmonic values (prev test = 3)

harmonics = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
fourier_error_rows = []

for k in harmonics:
    fourier_model = WeeklyChangeFourierRegressionModel(lookback_years=5, n_harmonics=k)
    fourier_result = evaluate_forecast(storage, fourier_model)
    metrics = error_metrics(fourier_result)["Model"]
    fourier_error_rows.append({"Harmonic": k, **metrics.to_dict()})

fourier_error_df = pd.DataFrame(fourier_error_rows)

print(fourier_error_df)
best_row = fourier_error_df.loc[fourier_error_df["MAE"].idxmin()]
print(f"Best harmonic (lowest MAE): k={int(best_row['Harmonic'])}, MAE={best_row['MAE']:.2f}")

fourier_error_long = fourier_error_df.melt(
    id_vars="Harmonic",
    value_vars=["MAE", "RMSE", "R2"],
    var_name="Metric",
    value_name="Value",
)
fig = px.line(
    fourier_error_long,
    x="Harmonic",
    y="Value",
    color="Metric",
    facet_row="Metric",
    markers=True,
    title="Fourier errors by harmonic count",
)
fig.update_yaxes(matches=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()


   Harmonic        MAE       RMSE        R2
0         1  61.463063  82.366796  0.580591
1         2  39.157334  57.368325  0.796540
2         3  39.490687  58.185891  0.790700
3         4  36.930872  55.651401  0.808536
4         5  36.564279  53.233945  0.824809
5         6  35.970116  51.536357  0.835804
6         7  34.132663  49.950249  0.845756
7         8  34.635607  50.556628  0.841988
8         9  35.371651  51.416164  0.836569
9        10  35.579261  53.435801  0.823478
Best harmonic (lowest MAE): k=7, MAE=34.13


In [62]:
# Fourier Regression (K=7 harmonics)
fourier_model_7 = WeeklyChangeFourierRegressionModel(lookback_years=5, n_harmonics=7)
fourier_result_7 = evaluate_forecast(storage, fourier_model_7)
fourier_result_7.head()

,date,week_of_year,weekly_change_bcf,predicted_weekly_change,lower_band,upper_band,forecast_deviation,outside_band
600,2026-01-02,1,-120.0,-132.601361,-172.963856,-92.238865,12.601361,False
601,2026-01-09,2,-70.0,-156.317586,-196.680082,-115.955091,86.317586,True
602,2026-01-16,3,-120.0,-180.192729,-220.555224,-139.830233,60.192729,True
603,2026-01-23,4,-241.0,-195.140193,-235.502689,-154.777698,-45.859807,True
604,2026-01-30,5,-359.0,-194.225826,-234.588321,-153.863330,-164.774174,True


In [63]:
plot_weekly_change_forecast(fourier_result_7, model_name=fourier_model_7.name).show()

In [64]:
# SARIMA
sarima_model = WeeklyChangeSARIMAModel(lookback_years=5)
sarima_result = evaluate_forecast(storage, sarima_model)
sarima_result.head()

,date,week_of_year,weekly_change_bcf,predicted_weekly_change,lower_band,upper_band,forecast_deviation,outside_band
600,2026-01-02,1,-120.0,-57.301105,-102.598369,-12.003841,-62.698895,True
601,2026-01-09,2,-70.0,-148.811460,-195.542252,-102.080668,78.811460,True
602,2026-01-16,3,-120.0,-169.242854,-216.021718,-122.463991,49.242854,True
603,2026-01-23,4,-241.0,-193.707199,-240.487700,-146.926697,-47.292801,True
604,2026-01-30,5,-359.0,-139.126345,-185.906902,-92.345788,-219.873655,True


In [65]:
plot_weekly_change_forecast(sarima_result, model_name=sarima_model.name).show()

In [ ]:


error_baseline = error_metrics(baseline_result).rename(columns={"Model": "5-Year Avg"})
error_lr = error_metrics(lr_result).rename(columns={"Model": "Linear Reg"})
error_fourier = error_metrics(fourier_result_7).rename(columns={"Model": "Fourier"})
error_sarima = error_metrics(sarima_result).rename(columns={"Model": "SARIMA"})

error_table = pd.concat([error_baseline, error_lr, error_fourier, error_sarima], axis=1)
error_table

# based on initial analysis, fourier model is best as it minimizes error metrics and maximizes r2

,5-Year Avg,Linear Reg,Fourier,SARIMA
MAE,39.680000,60.387824,34.132663,44.987191
RMSE,59.598819,80.494440,49.950249,65.625318
R2,0.780412,0.599442,0.845756,0.733758


Residual Analysis

In [67]:
# setup
models = [baseline_result, lr_result, fourier_result_7, sarima_result]

model_results = [
    (baseline_result, 'Residuals Over Time - 5-Year Average'),
    (lr_result, 'Residuals Over Time - Linear Regression'),
    (fourier_result_7, 'Residuals Over Time - Fourier Regression'),
    (sarima_result, 'Residuals Over Time - SARIMA')
]

In [68]:
#residuals over time


for result, title in model_results:



    fig = px.scatter(
        result,
        x='date',
        y='forecast_deviation',
        trendline='ols',
        title=title
    )
    fig.show()

    

In [69]:
# histogram (residuals over time)

for result, title in model_results:
    residual_hist = px.histogram(
        result,
        x='forecast_deviation',
        nbins=40,  # Increase number of bins for more granularity
        title=f'Histogram of Residuals - {title}',
        histnorm='probability density'  # Optional: to see density
    )
    residual_hist.update_traces(marker_line_width=1, marker_line_color="black")
    residual_hist.update_layout(
        bargap=0.1,
        xaxis_title="Forecast Deviation",
        yaxis_title="Density",
        template="plotly_white"
    )
    residual_hist.show()

    # models are generally left skewed, so investigate more

In [70]:
# actual vs forecast

for result, title in model_results:
    fig = px.scatter(
        result,
        x='predicted_weekly_change',
        y='weekly_change_bcf',
        title=f'Actual vs Forecast - {title.split(" - ")[1]}'
    )
    # Add 45 degree line (y=x)
    min_val = min(result['predicted_weekly_change'].min(), result['weekly_change_bcf'].min())
    max_val = max(result['predicted_weekly_change'].max(), result['weekly_change_bcf'].max())
    fig.add_shape(
        type="line",
        x0=min_val,
        y0=min_val,
        x1=max_val,
        y1=max_val,
        line=dict(color="red", dash="dash"),
        name="45 Degree Line"
    )
    fig.show()

# generally some systemically higher forecasting by models (i.e. actual < forecast), especially when there are large withdrawals (i.e. negative weekly_change_bcf)

In [71]:
# weekly residual analysis

for result, title in model_results:
    fig = px.scatter(
        result,
        x='week_of_year',
        y='forecast_deviation',
        title=f'Weekly Residuals - {title.split(" - ")[1]}'
    )
    fig.show()

# residuals generally centered around 0, but some systemic issues with models across early weeks. likely due to weather shocks/atypical market conditions. 

